# LC 102 — Binary Tree Level Order Traversal
**Difficulty:** Medium &nbsp;|&nbsp; **Category:** Trees / BFS
**Pattern:** BFS with Level Snapshot

<div style="border-left:4px solid purple; padding:10px 16px;
            background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> Snapshot `len(queue)` before
the inner loop — that number tells you exactly how many
nodes belong to the current level. Process only that
many, then move to the next level.
</div>

## Official Problem Statement

Given the `root` of a binary tree, return the
level-order traversal of its nodes' values
(i.e., from left to right, level by level).

**Example 1:**
```
    3
   / \
  9  20
    /  \
   15   7
```
```
Input:  root = [3,9,20,null,null,15,7]
Output: [[3],[9,20],[15,7]]
```
**Example 2:**
```
Input:  root = [1]
Output: [[1]]
```
**Example 3:**
```
Input:  root = []
Output: []
```

**Constraints:**
- `0 <= number of nodes <= 2000`
- `-1000 <= Node.val <= 1000`

## What This Is Actually Asking

Read the tree floor by floor, left to right.
Each floor's values go into their own list.
Return a list of those lists — one inner list per
floor of the tree.

## Walk Through an Example by Hand

```
Tree:  3
      / \
     9  20
       /  \
      15   7

Start: queue = [3]   result = []

Iteration 1 — level size = 1:
  pop 3  -> level = [3]
  push 9, push 20
  queue = [9, 20]  result = [[3]]

Iteration 2 — level size = 2:
  pop 9  -> level = [9],  no children
  pop 20 -> level = [9,20], push 15, push 7
  queue = [15, 7]  result = [[3],[9,20]]

Iteration 3 — level size = 2:
  pop 15 -> level = [15], no children
  pop 7  -> level = [15,7], no children
  queue = []  result = [[3],[9,20],[15,7]]

Queue empty -> done. Return [[3],[9,20],[15,7]]
```

## The Picture

```
The queue is a conveyor belt. Nodes ride it left
to right. When you process a node, its children
join the back of the belt for the NEXT round.

Level 0:  [ 3 ]                -> snapshot = 1
           pop 3, push 9 & 20

Level 1:  [ 9 | 20 ]           -> snapshot = 2
           pop 9 (no children)
           pop 20, push 15 & 7

Level 2:  [ 15 | 7 ]           -> snapshot = 2
           pop 15 & 7 (no children)

KEY: snapshot = len(queue) BEFORE inner loop.
Children added during this round don't get counted
until next round.
```

## When To Use This Pattern

- When you see **level-by-level**, **floor-by-floor**,
  or **layer** output, think **BFS with level snapshot**
- When you see **shortest path in a tree/graph**,
  think **BFS — it finds it naturally**
- When processing exactly one level, think
  **`level_size = len(queue)` before the inner loop**
- When you see **right/left side view**, think
  **same BFS — just take last/first element per level**

## The Approach

If the root is None, return an empty list.
Seed a queue with the root.
While the queue is non-empty, snapshot its current
size — that is the number of nodes at this level.
Pop exactly that many, collect their values, and
push their children.
Append the collected level to the result.

In [ ]:
from collections import deque   # O(1) popleft for the BFS queue
from typing import Optional, List

In [ ]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(vals: list) -> Optional[TreeNode]:
    """Build tree from level-order list (None = missing)."""
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue = deque([root])
    i = 1
    while queue and i < len(vals):
        node = queue.popleft()
        if i < len(vals) and vals[i] is not None:
            node.left = TreeNode(vals[i])
            queue.append(node.left)
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = TreeNode(vals[i])
            queue.append(node.right)
        i += 1
    return root


def test_harness(func):
    tests = [
        ([3,9,20,None,None,15,7], [[3],[9,20],[15,7]]),
        ([1],                     [[1]]),
        ([],                      []),
        ([1,2,3],                 [[1],[2,3]]),
        ([1,2,None,3],            [[1],[2],[3]]),
        ([1,None,2,None,3],       [[1],[2],[3]]),  # right skew
        ([1,2,3,4,5,6,7],         [[1],[2,3],[4,5,6,7]]),
    ]

    passed = 0
    for i, (vals, expected) in enumerate(tests):
        root = build_tree(vals)
        result = func(root)
        ok = result == expected
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        print(
            f"Test {i+1}: {status} | "
            f"expected={expected} | got={result}"
        )

    print(f"\n{passed}/{len(tests)} tests passed")

In [ ]:
def levelOrder(
    root: Optional[TreeNode]
) -> List[List[int]]:
    """
    Return level-order traversal as a list of lists.

    Seed a deque with root. Each outer iteration:
    snapshot level_size = len(queue). Process exactly
    level_size nodes, collect values, push children.
    Append collected level to result.

    Time:  O(n) — every node visited once
    Space: O(n) — queue holds at most one full level
                  (max width of the tree)
    """
    pass


# Quick debug — run this cell while building
t1 = build_tree([3,9,20,None,None,15,7])
t2 = build_tree([1])
t3 = build_tree([])
print(levelOrder(t1))  # [[3],[9,20],[15,7]]
print(levelOrder(t2))  # [[1]]
print(levelOrder(t3))  # []

In [ ]:
# Uncomment and run when solution is ready
# test_harness(levelOrder)

## Complexity

| Approach | Time | Space |
|---|---|---|
| DFS with depth tracking | O(n) | O(h) |
| BFS with level snapshot | O(n) | O(w) |

Both are O(n) — BFS is the natural fit because levels
map directly to BFS iterations; no depth bookkeeping
needed.

## Real World Connection

At Citi, the alert escalation system follows a tree:
Level 0 is the monitoring agent, Level 1 is the
on-call engineer, Level 2 is the team lead, Level 3
is the director.
Level-order traversal maps exactly to "who gets
paged first, who gets paged if they don't respond,
and who gets paged next" — processing one full
escalation tier before moving to the next.
The same BFS pattern drives the AWS service
dependency checker: validate all Tier-1 services
first, then Tier-2, then Tier-3 — level by level
before declaring a migration window safe.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra